In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from qubo.MaxCut import generate, solve_bruteforce
from quantum.backend import get_simulator
from quantum.circuit import build_brickwall, build_circuit
from quantum.hamiltonian import build_maxcut_hamiltonian, update_hamiltonian
from optimizer.adam_spsa import optimize
from ndar.run import sample_theta_full


# ======================================================================================
# CONFIG
# ======================================================================================
CONFIG = {
    # Problem / circuit
    "n_qubits": 10,
    "depth": 1,
    "edge_prob": 1.0,            # dense graph, closer to the paper's SK-like setting
    "weight_mode": "random",     # if you have bimodal +/-1, use that instead
    "seed": 1924,

    # Figure-4-left style experiment
    "n_instances": 10,           # paper uses 10 Hamiltonians
    "n_gauges": 10,              # paper uses 20 random gauges per Hamiltonian
    "shots": 256,                # simulated Fig. 4 left uses 256 samples
    "n_epochs": 10,             # paper uses 100 cost function evaluations
                                # here we map that idea to your optimizer budget

    # Optimizer
    "adam_lr": 0.10,
    "adam_beta1": 0.9,
    "adam_beta2": 0.999,
    "adam_eps": 1e-8,
    "spsa_c": 0.15,
    "spsa_gamma": 0.101,

    # Quantiles used to build the Y estimators from the sampled AR distribution
    # Simulated Fig. 4 left discusses:
    # - top 100%  -> 256 best samples
    # - top ~20%  -> 50 best samples
    # - top 0.4%  -> best sample
    "top_fracs": [1.0, 50 / 256, 1 / 256],

    # Output
    "out_dir": "fig4_left_vqa_like",
}


# ======================================================================================
# BASIC UTILS
# ======================================================================================
def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)


def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def random_bitstring(n, rng):
    return "".join(rng.choice(["0", "1"], size=n))


def adjacency_matrix_from_edges(n_qubits, edges):
    A = np.zeros((n_qubits, n_qubits), dtype=float)
    for i, j, w in edges:
        A[i, j] = w
        A[j, i] = w
    return A


def energy_of_bitstring(bitstring_q0_first, constant_term, zz_terms):
    """
    Same energy convention as your Hamiltonian builder.
    """
    z = np.array([1.0 if b == "0" else -1.0 for b in bitstring_q0_first], dtype=float)
    e = float(constant_term)
    for i, j, coeff in zz_terms:
        e += float(coeff) * z[i] * z[j]
    return float(e)


def ratio_of_bitstring(bitstring_q0_first, constant_term, zz_terms, E_GS_exact):
    """
    Same approximation-ratio-like quantity used in your current scripts:
        AR = E(bitstring) / E_GS_exact
    For minimization with negative E_GS, values closer to 1 are better.
    """
    return energy_of_bitstring(bitstring_q0_first, constant_term, zz_terms) / float(E_GS_exact)


def summarize_top_fracs(ratios, top_fracs):
    """
    Build AR estimators from quantiles of the sampled output distribution,
    after sorting descending, exactly in the spirit of the paper.
    """
    r = np.asarray(ratios, dtype=float)
    r = np.sort(r)[::-1]

    out = {}
    n = len(r)

    for frac in top_fracs:
        k = max(1, int(np.ceil(frac * n)))
        out[frac] = float(np.mean(r[:k]))

    out["best"] = float(r[0])
    out["mean"] = float(np.mean(r))
    return out


def rankdata_average(x):
    """
    Small scipy-free rankdata implementation with average ranks for ties.
    """
    x = np.asarray(x)
    order = np.argsort(x)
    ranks = np.empty(len(x), dtype=float)

    i = 0
    while i < len(x):
        j = i
        while j + 1 < len(x) and x[order[j + 1]] == x[order[i]]:
            j += 1
        avg_rank = 0.5 * (i + j) + 1.0
        for k in range(i, j + 1):
            ranks[order[k]] = avg_rank
        i = j + 1

    return ranks


def pearson_corr(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if len(x) < 2:
        return np.nan
    sx = np.std(x)
    sy = np.std(y)
    if sx == 0 or sy == 0:
        return np.nan

    return float(np.corrcoef(x, y)[0, 1])


def spearman_corr(x, y):
    return pearson_corr(rankdata_average(x), rankdata_average(y))


# ======================================================================================
# PROBLEM SETUP
# ======================================================================================
def build_problem(cfg, simulator, seed_instance):
    rng = np.random.default_rng(seed_instance)

    edges = generate(
        n_qubits=cfg["n_qubits"],
        edge_prob=cfg["edge_prob"],
        rng=rng,
        weight_mode=cfg["weight_mode"],
    )

    ansatz, params = build_brickwall(cfg["n_qubits"], cfg["depth"])
    transpiled_circuit = build_circuit(
        n_qubits=cfg["n_qubits"],
        ansatz=ansatz,
        simulator=simulator,
        seed=seed_instance,
    )

    _, H_constant, H_terms = build_maxcut_hamiltonian(cfg["n_qubits"], edges)

    E_GS_exact, exact_ground_states = solve_bruteforce(
        n_qubits=cfg["n_qubits"],
        constant_term=H_constant,
        zz_terms=H_terms,
    )

    # Keep same initialization for all gauges of the same Hamiltonian
    theta_init = rng.uniform(-0.1, 0.1, size=len(params))

    return {
        "edges": edges,
        "params": params,
        "transpiled_circuit": transpiled_circuit,
        "H_constant": H_constant,
        "H_terms": H_terms,
        "E_GS_exact": E_GS_exact,
        "exact_ground_states": exact_ground_states,
        "theta_init": theta_init,
    }


def run_single_vqa_final_distribution(
    *,
    label,
    theta_init,
    transpiled_circuit,
    params,
    simulator,
    shots,
    H_constant,
    H_terms,
    edges,
    E_GS_exact,
    seed_base,
    cfg,
):
    """
    Optimize once, then sample the final optimized distribution.
    This is the object we compare to the attractor AR.
    """
    train_result = optimize(
        label=label,
        theta_init=theta_init.copy(),
        transpiled_measured_template=transpiled_circuit,
        ansatz_params=params,
        simulator=simulator,
        shots=shots,
        H_constant_term=H_constant,
        H_zz_terms=H_terms,
        out_dir_sub=None,
        seed_base=seed_base,
        n_epochs=cfg["n_epochs"],
        lr=cfg["adam_lr"],
        beta1=cfg["adam_beta1"],
        beta2=cfg["adam_beta2"],
        eps=cfg["adam_eps"],
        spsa_c=cfg["spsa_c"],
        spsa_gamma=cfg["spsa_gamma"],
    )

    theta_best = train_result["theta_best"].copy()

    sample_result = sample_theta_full(
        n_qubits=cfg["n_qubits"],
        E_GS_exact=E_GS_exact,
        theta=theta_best,
        iter_label=label,
        transpiled_measured_template=transpiled_circuit,
        ansatz_params=params,
        simulator=simulator,
        shots=shots,
        seed_simulator=seed_base + 123456,
        edges=edges,
        H_constant_term=H_constant,
        H_zz_terms=H_terms,
        out_dir_sub=None,
    )

    return {
        "train_result": train_result,
        "sample_result": sample_result,
    }


# ======================================================================================
# FIGURE 4-LEFT STYLE EXPERIMENT
# ======================================================================================
def run_fig4_left_like_experiment(cfg, simulator, out_dir):
    """
    For each Hamiltonian instance:
      - generate a problem H
      - for 20 random bitflip gauges y:
          * build H_y
          * compute attractor AR = <y|H|y>/E_GS
          * optimize your VQA on H_y
          * sample final output distribution
          * build quantile estimators from sampled AR distribution

    This is the correct adaptation of the paper's left panel to your stack.
    """
    rows = []

    for inst_id in range(cfg["n_instances"]):
        print("=" * 100)
        print(f"INSTANCE {inst_id + 1}/{cfg['n_instances']}")
        print("=" * 100)

        seed_instance = cfg["seed"] + 10000 * inst_id
        rng_inst = np.random.default_rng(seed_instance)

        problem = build_problem(cfg, simulator, seed_instance)

        H0_constant = problem["H_constant"]
        H0_terms = problem["H_terms"]
        E_GS_exact = problem["E_GS_exact"]

        theta_init_shared = problem["theta_init"].copy()

        # Important:
        # The paper fixes the optimizer seed so that differences come mainly from
        # gauge choice + statistical noise + instance hardness.
        optimizer_seed_for_instance = cfg["seed"] + 500000 + inst_id

        # Save adjacency just for inspection
        A = adjacency_matrix_from_edges(cfg["n_qubits"], problem["edges"])
        pd.DataFrame(A).to_csv(out_dir / f"adjacency_instance_{inst_id:03d}.csv", index=False)

        for gauge_id in range(cfg["n_gauges"]):
            y = random_bitstring(cfg["n_qubits"], rng_inst)

            # Gauge-transformed Hamiltonian H_y
            _, Hy_constant, Hy_terms = update_hamiltonian(
                n_qubits=cfg["n_qubits"],
                constant_term=H0_constant,
                zz_terms=H0_terms,
                bitstring_q0_first=y,
            )

            # Attractor AR in transformed frame is the AR of |y> in original frame
            attractor_ratio = ratio_of_bitstring(
                bitstring_q0_first=y,
                constant_term=H0_constant,
                zz_terms=H0_terms,
                E_GS_exact=E_GS_exact,
            )

            res = run_single_vqa_final_distribution(
                label=f"inst_{inst_id}_gauge_{gauge_id}",
                theta_init=theta_init_shared,
                transpiled_circuit=problem["transpiled_circuit"],
                params=problem["params"],
                simulator=simulator,
                shots=cfg["shots"],
                H_constant=Hy_constant,
                H_terms=Hy_terms,
                edges=problem["edges"],
                E_GS_exact=E_GS_exact,
                seed_base=optimizer_seed_for_instance,
                cfg=cfg,
            )

            sample_result = res["sample_result"]
            ratios = np.asarray(sample_result["ratios"], dtype=float)

            summary = summarize_top_fracs(ratios, cfg["top_fracs"])

            row = {
                "instance_id": inst_id,
                "gauge_id": gauge_id,
                "gauge_bitstring": y,
                "attractor_ratio": attractor_ratio,
                "optimized_ratio_mean_all": summary["mean"],
                "optimized_ratio_best_sample": summary["best"],
                "expected_ratio_transformed": sample_result.get("expected_ratio", np.nan),
                "zero_ratio_transformed": sample_result.get("zero_ratio", np.nan),
            }

            for frac in cfg["top_fracs"]:
                frac_key = f"optimized_ratio_topfrac_{str(frac).replace('.', 'p')}"
                row[frac_key] = summary[frac]

            rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(out_dir / "fig4_left_like_data.csv", index=False)
    return df


# ======================================================================================
# PLOTS
# ======================================================================================
def plot_fig4_left_like(df, cfg, out_dir):
    """
    Build 3 panels, like the paper's left-side logic:
      - top 100%
      - top ~20%
      - best sample
    """
    frac_cols = []
    frac_labels = []

    for frac in cfg["top_fracs"]:
        col = f"optimized_ratio_topfrac_{str(frac).replace('.', 'p')}"
        frac_cols.append(col)

        if abs(frac - 1.0) < 1e-12:
            frac_labels.append("Top 100%")
        elif abs(frac - 50 / 256) < 1e-12:
            frac_labels.append("Top ~20%")
        elif abs(frac - 1 / 256) < 1e-12:
            frac_labels.append("Best sample")
        else:
            frac_labels.append(f"Top {100 * frac:.2f}%")

    x = df["attractor_ratio"].to_numpy()

    ymins = [df[c].min() for c in frac_cols]
    ymaxs = [df[c].max() for c in frac_cols]

    global_min = float(min(x.min(), min(ymins)))
    global_max = float(max(x.max(), max(ymaxs)))
    pad = 0.03 * max(1e-9, global_max - global_min)

    xmin = global_min - pad
    xmax = global_max + pad

    fig, axes = plt.subplots(1, len(frac_cols), figsize=(5.6 * len(frac_cols), 5.2), sharex=True, sharey=True)
    if len(frac_cols) == 1:
        axes = [axes]

    for ax, col, label in zip(axes, frac_cols, frac_labels):
        y = df[col].to_numpy()

        hist = ax.hist2d(
            x, y,
            bins=25,
            range=[[xmin, xmax], [xmin, xmax]],
            cmap="plasma"
        )
        fig.colorbar(hist[3], ax=ax, label="Count")

        # Shade region where optimization did not improve over attractor
        xx = np.linspace(xmin, xmax, 300)
        ax.fill_between(xx, xmin, xx, color="gray", alpha=0.15)
        ax.plot(xx, xx, "k--", linewidth=1.0)

        rp = pearson_corr(x, y)
        rs = spearman_corr(x, y)

        ax.set_title(f"{label}\nPearson={rp:.3f}, Spearman={rs:.3f}")
        ax.set_xlabel("Attractor AR")
        ax.set_ylabel("Optimized AR")
        ax.grid(alpha=0.2)

    plt.suptitle("Figure-4-left style: VQA vs attractor state", fontsize=14)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(out_dir / "fig4_left_like_hist2d.png", dpi=220, bbox_inches="tight")
    plt.close()


def plot_fig4_left_like_scatter(df, cfg, out_dir):
    """
    Cleaner scatter version for the ~20% panel.
    """
    frac = 50 / 256
    col = f"optimized_ratio_topfrac_{str(frac).replace('.', 'p')}"

    x = df["attractor_ratio"].to_numpy()
    y = df[col].to_numpy()

    rp = pearson_corr(x, y)
    rs = spearman_corr(x, y)

    global_min = float(min(x.min(), y.min()))
    global_max = float(max(x.max(), y.max()))
    pad = 0.03 * max(1e-9, global_max - global_min)

    xmin = global_min - pad
    xmax = global_max + pad
    xx = np.linspace(xmin, xmax, 300)

    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    ax.scatter(x, y, s=35, alpha=0.75)
    ax.fill_between(xx, xmin, xx, color="gray", alpha=0.15)
    ax.plot(xx, xx, "k--", linewidth=1.0)

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(xmin, xmax)
    ax.set_xlabel("Attractor AR")
    ax.set_ylabel("Optimized AR (Top ~20%)")
    ax.set_title(f"Scatter version\nPearson={rp:.3f}, Spearman={rs:.3f}")
    ax.grid(alpha=0.25)

    plt.tight_layout()
    plt.savefig(out_dir / "fig4_left_like_scatter_top20.png", dpi=220, bbox_inches="tight")
    plt.close()


# ======================================================================================
# MAIN
# ======================================================================================
def main():
    cfg = CONFIG.copy()

    out_dir = Path(cfg["out_dir"])
    ensure_dir(out_dir)
    save_json(cfg, out_dir / "config.json")

    simulator = get_simulator(n_qubits=cfg["n_qubits"])

    df = run_fig4_left_like_experiment(cfg, simulator, out_dir)

    plot_fig4_left_like(df, cfg, out_dir)
    plot_fig4_left_like_scatter(df, cfg, out_dir)

    # Summary by panel
    summary_rows = []
    for frac in cfg["top_fracs"]:
        col = f"optimized_ratio_topfrac_{str(frac).replace('.', 'p')}"
        rp = pearson_corr(df["attractor_ratio"], df[col])
        rs = spearman_corr(df["attractor_ratio"], df[col])

        summary_rows.append({
            "panel_frac": frac,
            "panel_col": col,
            "pearson": rp,
            "spearman": rs,
            "mean_attractor_ratio": float(df["attractor_ratio"].mean()),
            "mean_optimized_ratio": float(df[col].mean()),
        })

    df_summary = pd.DataFrame(summary_rows)
    df_summary.to_csv(out_dir / "fig4_left_like_summary.csv", index=False)

    summary = {
        "data_csv": str(out_dir / "fig4_left_like_data.csv"),
        "summary_csv": str(out_dir / "fig4_left_like_summary.csv"),
        "hist2d_plot": str(out_dir / "fig4_left_like_hist2d.png"),
        "scatter_plot": str(out_dir / "fig4_left_like_scatter_top20.png"),
    }
    save_json(summary, out_dir / "summary.json")

    print("=" * 100)
    print("DONE")
    print("=" * 100)
    print(json.dumps(summary, indent=2, ensure_ascii=False))


if __name__ == "__main__":
    main()

INSTANCE 1/10
inst_0_gauge_0 | epoch=001 | loss=-27.821378 | best=-27.821378 | grad_norm=0.487694
inst_0_gauge_0 | epoch=002 | loss=-27.908787 | best=-27.908787 | grad_norm=0.353280
inst_0_gauge_0 | epoch=003 | loss=-27.976762 | best=-27.976762 | grad_norm=0.625831
inst_0_gauge_0 | epoch=004 | loss=-27.983133 | best=-27.983133 | grad_norm=3.844501
inst_0_gauge_0 | epoch=005 | loss=-28.062929 | best=-28.062929 | grad_norm=4.496607
inst_0_gauge_0 | epoch=006 | loss=-28.176126 | best=-28.176126 | grad_norm=6.591449
inst_0_gauge_0 | epoch=007 | loss=-28.377748 | best=-28.377748 | grad_norm=1.916787
inst_0_gauge_0 | epoch=008 | loss=-28.534653 | best=-28.534653 | grad_norm=2.286889
inst_0_gauge_0 | epoch=009 | loss=-28.730398 | best=-28.730398 | grad_norm=1.335514
inst_0_gauge_0 | epoch=010 | loss=-28.772184 | best=-28.772184 | grad_norm=3.074071
inst_0_gauge_1 | epoch=001 | loss=-32.844443 | best=-32.844443 | grad_norm=0.263819
inst_0_gauge_1 | epoch=002 | loss=-32.815349 | best=-32.844443

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


def plot_right_panel_like_fig4(
    csv_path,
    out_path=None,
    title="Right-panel style: VQA+NDAR vs VQA vs Random"
):
    csv_path = Path(csv_path)
    df = pd.read_csv(csv_path)

    required_cols = {"instance_id", "method", "iteration", "best_ratio_cumulative"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns in CSV: {sorted(missing)}")

    # Standardize method names
    method_map = {
        "NDAR": "VQA+NDAR",
        "VQA+NDAR": "VQA+NDAR",
        "Plain": "VQA",
        "PLAIN": "VQA",
        "VQA": "VQA",
        "Random": "Random",
        "RANDOM": "Random",
    }
    df["method"] = df["method"].map(lambda x: method_map.get(str(x), str(x)))

    order = ["VQA+NDAR", "VQA", "Random"]
    df = df[df["method"].isin(order)].copy()

    if df.empty:
        raise ValueError("No recognized methods found in CSV.")

    styles = {
        "VQA+NDAR": {
            "color": "darkorange",
            "instance_ls": ":",
            "instance_alpha": 0.45,
            "instance_lw": 1.4,
        },
        "VQA": {
            "color": "cornflowerblue",
            "instance_ls": ":",
            "instance_alpha": 0.45,
            "instance_lw": 1.4,
        },
        "Random": {
            "color": "gray",
            "instance_ls": ":",
            "instance_alpha": 0.45,
            "instance_lw": 1.2,
        },
    }

    fig, ax = plt.subplots(figsize=(10, 6))

    for method in order:
        dmethod = df[df["method"] == method].copy()
        if dmethod.empty:
            continue

        # Thin dotted per-instance lines
        for inst_id in sorted(dmethod["instance_id"].unique()):
            d = dmethod[dmethod["instance_id"] == inst_id].sort_values("iteration")
            ax.plot(
                d["iteration"],
                d["best_ratio_cumulative"],
                linestyle=styles[method]["instance_ls"],
                color=styles[method]["color"],
                alpha=styles[method]["instance_alpha"],
                linewidth=styles[method]["instance_lw"],
            )

        # Min / max solid lines across instances
        pivot = dmethod.pivot(
            index="iteration",
            columns="instance_id",
            values="best_ratio_cumulative"
        ).sort_index()

        y_min = pivot.min(axis=1)
        y_max = pivot.max(axis=1)

        ax.plot(
            pivot.index,
            y_min,
            color=styles[method]["color"],
            linewidth=2.5,
            label=f"{method} min/max",
        )
        ax.plot(
            pivot.index,
            y_max,
            color=styles[method]["color"],
            linewidth=2.5,
        )

    ax.axhline(1.0, color="black", linestyle="--", linewidth=1.0)
    ax.set_xlabel("Iteration / matched cumulative budget index")
    ax.set_ylabel(r"Best-found approximation ratio $(E/E_{GS})$")
    ax.set_title(title)
    ax.set_ylim(0.0, 1.05)
    ax.grid(alpha=0.25)
    ax.legend()
    plt.tight_layout()

    if out_path is None:
        out_path = csv_path.with_name("right_panel_like_fig4.png")
    else:
        out_path = Path(out_path)

    plt.savefig(out_path, dpi=220, bbox_inches="tight")
    plt.close()

    print(f"Saved plot to: {out_path}")


# Example use:
plot_right_panel_like_fig4(
    csv_path="ndar_vqa_minimal_benchmark/benchmark_data.csv",
    out_path="ndar_vqa_minimal_benchmark/right_panel_like_fig4.png"
)

Saved plot to: ndar_vqa_minimal_benchmark/right_panel_like_fig4.png
